<a href="https://colab.research.google.com/github/sara-iqbal/TikTok-Shop-Conversion-Optimization-Engine/blob/main/TikTok_Shop_Conversion_Optimization_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

TikTok Shop Conversion Optimization Engine

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.power import TTestIndPower
import statsmodels.api as sm

# 1. Power Analysis: How many users do we need?
effect_size = 0.02  # We want to detect a 2% lift
alpha = 0.05       # 5% significance level
power = 0.8        # 80% power

analysis = TTestIndPower()
required_n = analysis.solve_power(effect_size=effect_size, power=power, alpha=alpha, ratio=1.0)
print(f"Required Sample Size per Group: {int(required_n)}")

# 2. Synthetic Data Generation (100,000 sessions)
np.random.seed(42)
n_samples = 100000

data = pd.DataFrame({
    'user_id': range(n_samples),
    'group': np.random.choice(['Control', 'Variant'], n_samples),
    'age_group': np.random.choice(['Gen Z', 'Millennial'], n_samples, p=[0.6, 0.4]),
    'days_since_signup': np.random.randint(1, 365, n_samples)
})

# Simulate Conversion (Variant B gets a 2.4% lift)
def simulate_conversion(group):
    base_rate = 0.05
    lift = 0.024 if group == 'Variant' else 0.0
    return np.random.binomial(1, base_rate + lift)

# Simulate Spend (Non-normal distribution for Mann-Whitney U)
def simulate_spend(converted):
    if converted == 0: return 0
    return np.random.lognormal(mean=2.5, sigma=0.8)

data['converted'] = data['group'].apply(simulate_conversion)
data['order_value'] = data['converted'].apply(simulate_spend)

# 3. Hypothesis Testing (Conversion Rate)
conversions = data.groupby('group')['converted'].agg(['sum', 'count'])
z_score, p_value = sm.stats.proportions_ztest(conversions['sum'], conversions['count'])

# 4. AOV Testing (Mann-Whitney U for non-normal spend)
control_spend = data[(data['group'] == 'Control') & (data['converted'] == 1)]['order_value']
variant_spend = data[(data['group'] == 'Variant') & (data['converted'] == 1)]['order_value']
u_stat, u_p_value = stats.mannwhitneyu(control_spend, variant_spend)

# 5. Output Results
print(f"Conversion Z-Test P-Value: {p_value:.4f}")
print(f"AOV Mann-Whitney P-Value: {u_p_value:.4f}")

Required Sample Size per Group: 39245
Conversion Z-Test P-Value: 0.0000
AOV Mann-Whitney P-Value: 0.4753
